Importação das bibliotecas

In [1]:
import pandas as pd
import os

Carregamento dos dados da camada Bronze

In [2]:
# Definindo os caminhos para as camadas Bronze e Silver
raw_path = '../Data-Layer/raw/' 
silver_path = '../Data-Layer/silver/' 


# Nome do arquivo de origem
raw_file = 'lancamentos-comerciais-por-distribuidoras.csv'

# Criando o caminho completo para o arquivoelo
file_path = os.path.join(raw_path, raw_file)

# Carregando o dataset
try:
    df = pd.read_csv(file_path, sep=';', encoding='utf-8')
    print("Arquivo CSV bruto carregado com sucesso!")
    print(f"O DataFrame tem {df.shape[0]} linhas e {df.shape[1]} colunas.")
except FileNotFoundError:
    print(f"Erro: Arquivo '{file_path}' não encontrado. Certifique-se de que ele está na pasta 'raw'.")
    df = None

# Visualizando as primeiras linhas do dataframe
if df is not None:
    display(df.head())

Arquivo CSV bruto carregado com sucesso!
O DataFrame tem 6653 linhas e 10 colunas.


,DATA_LANCAMENTO_OBRA,TITULO_ORIGINAL,CPB_ROE,TIPO_OBRA,PAIS_OBRA,PUBLICO_TOTAL,RENDA_TOTAL,RAZAO_SOCIAL_DISTRIBUIDORA,REGISTRO_DISTRIBUIDORA,CNPJ_DISTRIBUIDORA
0,31/07/2025,ALDO BALDIN - UMA VIDA PELA MÚSICA,B2400467800000,DOCUMENTÁRIO,BRASIL,15,"R$ 396,34",BRETZ FILMES DISTRIBUIDORA E PRODUTORA LTDA - EPP,19243.0,39.079.678/0001-47
1,31/07/2025,DEATH OF A UNICORN,E2500223800000,FICÇÃO,ESTADOS UNIDOS,245,"R$ 5.725,40",WARNER BROS. (SOUTH) INC.,265.0,33.015.827/0001-28
2,31/07/2025,GUNS UP,E2500153200000,FICÇÃO,ESTADOS UNIDOS,994,"R$ 18.646,57",DIAMOND FILMS DO BRASIL PRODUÇÃO E DISTRIBUIÇÃ...,22724.0,17.095.184/0001-13
3,31/07/2025,MATERIALISTS,E2500150300000,FICÇÃO,ESTADOS UNIDOS,37310,"R$ 851.099,53",COLUMBIA TRISTAR FILMES DO BRASIL LTDA,84.0,00.979.601/0001-98
4,31/07/2025,NADA,B2300179900000,FICÇÃO,BRASIL,238,"R$ 311,31",EMBAUBA FILMES LTDA,23638.0,15.144.532/0001-70


Dicionário de Dados e Renomeação das Colunas

In [3]:
# Dicionário para renomear as colunas
rename_dict = {
    'DATA_LANCAMENTO_OBRA': 'data_lancamento',
    'TITULO_ORIGINAL': 'titulo_original',
    'CPB_ROE': 'cpb_roe',
    'TIPO_OBRA': 'tipo_obra',
    'PAIS_OBRA': 'pais_obra',
    'PUBLICO_TOTAL': 'publico_total',
    'RENDA_TOTAL': 'renda_total',
    'RAZAO_SOCIAL_DISTRIBUIDORA': 'distribuidora',
    'REGISTRO_DISTRIBUIDORA': 'registro_distribuidora',
    'CNPJ_DISTRIBUIDORA': 'cnpj_distribuidora'
}

df_renamed = df.rename(columns=rename_dict)

print("Colunas renomeadas com sucesso!")
display(df_renamed.head())

Colunas renomeadas com sucesso!


,data_lancamento,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,distribuidora,registro_distribuidora,cnpj_distribuidora
0,31/07/2025,ALDO BALDIN - UMA VIDA PELA MÚSICA,B2400467800000,DOCUMENTÁRIO,BRASIL,15,"R$ 396,34",BRETZ FILMES DISTRIBUIDORA E PRODUTORA LTDA - EPP,19243.0,39.079.678/0001-47
1,31/07/2025,DEATH OF A UNICORN,E2500223800000,FICÇÃO,ESTADOS UNIDOS,245,"R$ 5.725,40",WARNER BROS. (SOUTH) INC.,265.0,33.015.827/0001-28
2,31/07/2025,GUNS UP,E2500153200000,FICÇÃO,ESTADOS UNIDOS,994,"R$ 18.646,57",DIAMOND FILMS DO BRASIL PRODUÇÃO E DISTRIBUIÇÃ...,22724.0,17.095.184/0001-13
3,31/07/2025,MATERIALISTS,E2500150300000,FICÇÃO,ESTADOS UNIDOS,37310,"R$ 851.099,53",COLUMBIA TRISTAR FILMES DO BRASIL LTDA,84.0,00.979.601/0001-98
4,31/07/2025,NADA,B2300179900000,FICÇÃO,BRASIL,238,"R$ 311,31",EMBAUBA FILMES LTDA,23638.0,15.144.532/0001-70


Processo de Limpeza e Transformação

In [4]:
df_processed = df_renamed.copy()

# 1. Tratamento da coluna 'data_lancamento'
# Converte para data. Se não conseguir, o valor se tornará 'NaT' (Not a Time)
df_processed['data_lancamento'] = pd.to_datetime(df_processed['data_lancamento'], format='%d/%m/%Y', errors='coerce')

# Critério de insucesso: data_lancamento é nula (NaT)
insucesso_mask = df_processed['data_lancamento'].isnull()

df_insucesso = df_processed[insucesso_mask].copy()
df_sucesso = df_processed[~insucesso_mask].copy()

print(f"Total de registros: {len(df_processed)}")
print(f"Registros de sucesso (com data válida): {len(df_sucesso)}")
print(f"Registros de insucesso (sem data válida): {len(df_insucesso)}")

# 2. Tratamento da coluna 'renda_total'
df_sucesso['renda_total'] = df_sucesso['renda_total'].replace({'R\$ ': '', '\.': ''}, regex=True).str.replace(',', '.')
df_sucesso['renda_total'] = pd.to_numeric(df_sucesso['renda_total'], errors='coerce').fillna(0)

# 3. Tratamento da coluna 'publico_total'
df_sucesso['publico_total'] = pd.to_numeric(df_sucesso['publico_total'], errors='coerce').fillna(0).astype(int)

# 4. Feature Engineering: Criando colunas de Ano, Mês e Dia
df_sucesso['ano_lancamento'] = df_sucesso['data_lancamento'].dt.year
df_sucesso['mes_lancamento'] = df_sucesso['data_lancamento'].dt.month
df_sucesso['dia_lancamento'] = df_sucesso['data_lancamento'].dt.day

# 5. Padronização de colunas de texto - manter uppercase para compatibilidade com Gold
text_columns = ['titulo_original', 'tipo_obra', 'pais_obra', 'distribuidora']
for col in text_columns:
    df_sucesso[col] = df_sucesso[col].str.strip().str.upper()
    
# 6. Tratamento de valores ausentes em colunas de texto
string_columns = ['titulo_original', 'cpb_roe', 'tipo_obra', 'pais_obra', 'distribuidora', 'cnpj_distribuidora']
for col in string_columns:
    df_sucesso[col] = df_sucesso[col].fillna('NÃO INFORMADO')
    
# 7. Garantir o tipo de dado e padronização para campos chave
df_sucesso['registro_distribuidora'] = df_sucesso['registro_distribuidora'].astype(str).str.strip()
df_sucesso['cnpj_distribuidora'] = df_sucesso['cnpj_distribuidora'].str.strip()
df_sucesso['cpb_roe'] = df_sucesso['cpb_roe'].str.strip()

print("\nLimpeza e transformação dos dados de SUCESSO concluídas.")
display(df_sucesso.head())

print("\nDados de INSUCESSO (registros que não serão processados):")
display(df_insucesso)

Total de registros: 6653
Registros de sucesso (com data válida): 6653
Registros de insucesso (sem data válida): 0

Limpeza e transformação dos dados de SUCESSO concluídas.


<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\.'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_44426/1206174614.py:18: SyntaxWarning: invalid escape sequence '\$'
  df_sucesso['renda_total'] = df_sucesso['renda_total'].replace({'R\$ ': '', '\.': ''}, regex=True).str.replace(',', '.')
/tmp/ipykernel_44426/1206174614.py:18: SyntaxWarning: invalid escape sequence '\.'
  df_sucesso['renda_total'] = df_sucesso['renda_total'].replace({'R\$ ': '', '\.': ''}, regex=True).str.replace(',', '.')


,data_lancamento,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,distribuidora,registro_distribuidora,cnpj_distribuidora,ano_lancamento,mes_lancamento,dia_lancamento
0,2025-07-31,ALDO BALDIN - UMA VIDA PELA MÚSICA,B2400467800000,DOCUMENTÁRIO,BRASIL,15,396.34,BRETZ FILMES DISTRIBUIDORA E PRODUTORA LTDA - EPP,19243.0,39.079.678/0001-47,2025,7,31
1,2025-07-31,DEATH OF A UNICORN,E2500223800000,FICÇÃO,ESTADOS UNIDOS,245,5725.40,WARNER BROS. (SOUTH) INC.,265.0,33.015.827/0001-28,2025,7,31
2,2025-07-31,GUNS UP,E2500153200000,FICÇÃO,ESTADOS UNIDOS,994,18646.57,DIAMOND FILMS DO BRASIL PRODUÇÃO E DISTRIBUIÇÃ...,22724.0,17.095.184/0001-13,2025,7,31
3,2025-07-31,MATERIALISTS,E2500150300000,FICÇÃO,ESTADOS UNIDOS,37310,851099.53,COLUMBIA TRISTAR FILMES DO BRASIL LTDA,84.0,00.979.601/0001-98,2025,7,31
4,2025-07-31,NADA,B2300179900000,FICÇÃO,BRASIL,238,311.31,EMBAUBA FILMES LTDA,23638.0,15.144.532/0001-70,2025,7,31



Dados de INSUCESSO (registros que não serão processados):


,data_lancamento,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,distribuidora,registro_distribuidora,cnpj_distribuidora


Detecção de Outliers

In [5]:
# --- IDENTIFICANDO OUTLIERS DE RENDA E PÚBLICO ---

# Usaremos o método IQR (Interquartile Range) para definir o que é um outlier.
# Um valor é considerado outlier se for maior que Q3 + 1.5 * IQR.

# Calculando Q1, Q3 e IQR para 'renda_total'
Q1_renda = df_sucesso['renda_total'].quantile(0.25)
Q3_renda = df_sucesso['renda_total'].quantile(0.75)
IQR_renda = Q3_renda - Q1_renda
limite_renda = Q3_renda + 1.5 * IQR_renda

# Calculando Q1, Q3 e IQR para 'publico_total'
Q1_publico = df_sucesso['publico_total'].quantile(0.25)
Q3_publico = df_sucesso['publico_total'].quantile(0.75)
IQR_publico = Q3_publico - Q1_publico
limite_publico = Q3_publico + 1.5 * IQR_publico

print(f"Limite para outlier de renda: R$ {limite_renda:,.2f}")
print(f"Limite para outlier de público: {limite_publico:,.0f} pessoas")

# Criando a coluna 'is_outlier'
df_sucesso['is_outlier'] = (df_sucesso['renda_total'] > limite_renda) | (df_sucesso['publico_total'] > limite_publico)

print(f"\nEncontrados {df_sucesso['is_outlier'].sum()} outliers.")
display(df_sucesso[df_sucesso['is_outlier'] == True].head())

Limite para outlier de renda: R$ 3,839,994.74
Limite para outlier de público: 266,837 pessoas

Encontrados 1219 outliers.


,data_lancamento,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,distribuidora,registro_distribuidora,cnpj_distribuidora,ano_lancamento,mes_lancamento,dia_lancamento,is_outlier
11,2025-07-24,THE FANTASTIC FOUR: FIRST STEPS,E2500247600000,FICÇÃO,ESTADOS UNIDOS,1817536,38680764.21,THE WALT DISNEY COMPANY (BRASIL) LTDA.,18398.0,73.042.962/0001-87,2025,7,24,True
21,2025-07-17,I KNOW WHAT YOU DID LAST SUMMER,E2500150200000,FICÇÃO,ESTADOS UNIDOS,355466,7007182.31,COLUMBIA TRISTAR FILMES DO BRASIL LTDA,84.0,00.979.601/0001-98,2025,7,17,True
24,2025-07-17,SMURFS,E2500232800000,ANIMAÇÃO,ESTADOS UNIDOS,1051755,19750416.52,PARAMOUNT PICTURES BRASIL DISTRIBUIDORA DE FIL...,169.0,27.654.722/0001-16,2025,7,17,True
28,2025-07-10,SUPERMAN,E2500221500000,FICÇÃO,ESTADOS UNIDOS,3737773,80429459.44,WARNER BROS. (SOUTH) INC.,265.0,33.015.827/0001-28,2025,7,10,True
33,2025-07-03,JURASSIC WORLD: REBIRTH,E2500212900000,FICÇÃO,ESTADOS UNIDOS,2927769,60375537.06,WARNER BROS. (SOUTH) INC.,265.0,33.015.827/0001-28,2025,7,3,True


In [6]:
# --- IDENTIFICANDO OUTLIERS DE TICKET MÉDIO POR PESSOA ---

# Calculando o ticket médio por pessoa (renda_total / publico_total)
# Evitando divisão por zero - se público = 0, ticket médio = 0
df_sucesso['ticket_medio'] = df_sucesso.apply(
    lambda row: row['renda_total'] / row['publico_total'] if row['publico_total'] > 0 else 0, 
    axis=1
)

print("Estatísticas descritivas do ticket médio:")
print(df_sucesso['ticket_medio'].describe())

# Aplicando o método IQR para identificar outliers no ticket médio
Q1_ticket = df_sucesso['ticket_medio'].quantile(0.25)
Q3_ticket = df_sucesso['ticket_medio'].quantile(0.75)
IQR_ticket = Q3_ticket - Q1_ticket

# Definindo limites inferior e superior para outliers
limite_inferior_ticket = Q1_ticket - 1.5 * IQR_ticket
limite_superior_ticket = Q3_ticket + 1.5 * IQR_ticket

print(f"\nAnálise do Ticket Médio:")
print(f"Q1 (25%): R$ {Q1_ticket:.2f}")
print(f"Q3 (75%): R$ {Q3_ticket:.2f}")
print(f"IQR: R$ {IQR_ticket:.2f}")
print(f"Limite inferior para outlier: R$ {limite_inferior_ticket:.2f}")
print(f"Limite superior para outlier: R$ {limite_superior_ticket:.2f}")

# Criando coluna para identificar outliers de ticket médio
df_sucesso['is_outlier_ticket'] = (
    (df_sucesso['ticket_medio'] < limite_inferior_ticket) | 
    (df_sucesso['ticket_medio'] > limite_superior_ticket)
)

# Contando outliers
outliers_ticket_count = df_sucesso['is_outlier_ticket'].sum()
print(f"\nEncontrados {outliers_ticket_count} outliers de ticket médio.")

# Separando outliers inferiores e superiores
outliers_baixo = df_sucesso[df_sucesso['ticket_medio'] < limite_inferior_ticket]
outliers_alto = df_sucesso[df_sucesso['ticket_medio'] > limite_superior_ticket]

print(f"Outliers com ticket médio BAIXO: {len(outliers_baixo)}")
print(f"Outliers com ticket médio ALTO: {len(outliers_alto)}")

# Mostrando alguns exemplos de outliers
if len(outliers_alto) > 0:
    print("\n--- OUTLIERS COM TICKET MÉDIO ALTO ---")
    display(outliers_alto[['titulo_original', 'renda_total', 'publico_total', 'ticket_medio']].head())

if len(outliers_baixo) > 0:
    print("\n--- OUTLIERS COM TICKET MÉDIO BAIXO ---")
    display(outliers_baixo[['titulo_original', 'renda_total', 'publico_total', 'ticket_medio']].head())

Estatísticas descritivas do ticket médio:
count    6653.000000
mean       14.090720
std         5.044037
min         0.643038
25%        10.569000
50%        13.658295
75%        16.798050
max        65.000000
Name: ticket_medio, dtype: float64

Análise do Ticket Médio:
Q1 (25%): R$ 10.57
Q3 (75%): R$ 16.80
IQR: R$ 6.23
Limite inferior para outlier: R$ 1.23
Limite superior para outlier: R$ 26.14

Encontrados 129 outliers de ticket médio.
Outliers com ticket médio BAIXO: 6
Outliers com ticket médio ALTO: 123

--- OUTLIERS COM TICKET MÉDIO ALTO ---


,titulo_original,renda_total,publico_total,ticket_medio
0,ALDO BALDIN - UMA VIDA PELA MÚSICA,396.34,15,26.422667
6,BTS ARMY: FOREVER WE ARE YOUNG,74580.61,2273,32.811531
15,ROGER WATERS THIS IS NOT A DRILL LIVE FROM PRA...,120638.50,3789,31.839140
35,ATEEZ WORLD TOUR TOWARDS THE LIGHT WILL TO POWER,423422.19,12695,33.353461
65,J-HOPE TOUR 'HOPE ON THE STAGE' IN JAPAN: LIVE...,2366326.65,63087,37.508942



--- OUTLIERS COM TICKET MÉDIO BAIXO ---


,titulo_original,renda_total,publico_total,ticket_medio
83,OS DRAGÕES,480.0,480,1.0
84,OS DRAGÕES,127.0,127,1.0
92,CAIAM AS ROSAS BRANCAS!,1.0,1,1.0
2486,LAS HIJAS DEL FUEGO,1.0,1,1.0
4943,SÃO SILVESTRE,1.0,1,1.0


Salvando o arquivo processado no banco

In [7]:
import time
import os
import psycopg2
import pandas as pd

print("Estabelecendo conexão com o banco de dados...")
time.sleep(5)  # aguarda o banco inicializar um pouco

DB_HOST = os.getenv("DB_HOST", os.getenv("DATABASE_HOST", "localhost"))       # 'db' para Docker; use 'localhost' ao rodar local
DB_PORT = os.getenv("DB_PORT", os.getenv("DATABASE_PORT", "5433"))   
DB_USER = os.getenv("DB_USER", os.getenv("DATABASE_USER", "user"))
DB_PASSWORD = os.getenv("DB_PASSWORD", os.getenv("DATABASE_PASSWORD", "password"))
DB_NAME = os.getenv("DB_NAME", os.getenv("DATABASE_NAME", "filmes_db"))

try:
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        dbname=DB_NAME
    )
    conn.autocommit = True
    cur = conn.cursor()
    
    # Executa DDL (ddl.sql) para garantir que a tabela exista
    ddl_file_path = '../Data-Layer/silver/ddl.sql'
    
    print(f"Carregando DDL de: {ddl_file_path}")
    with open(ddl_file_path, 'r', encoding='utf-8') as file:
        ddl_script = file.read()
    
    print("Executando DDL para criar tabela 'lancamentos'...")
    cur.execute(ddl_script)
    print("✓ Tabela 'lancamentos' criada/verificada com sucesso!")
    print("✓ Índices criados com sucesso!")

    insert_sql = """
    INSERT INTO lancamentos (
        data_lancamento, titulo_original, cpb_roe, tipo_obra, pais_obra,
        publico_total, renda_total, distribuidora, registro_distribuidora,
        cnpj_distribuidora, ano_lancamento, mes_lancamento, dia_lancamento, is_outlier
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    print("Carregando dados de sucesso no banco...")
    inserted = 0
    for _, row in df_sucesso.iterrows():
        vals = (
            row['data_lancamento'].date() if pd.notnull(row['data_lancamento']) else None,
            None if pd.isna(row.get('titulo_original')) else str(row.get('titulo_original')),
            None if pd.isna(row.get('cpb_roe')) else str(row.get('cpb_roe')),
            None if pd.isna(row.get('tipo_obra')) else str(row.get('tipo_obra')),
            None if pd.isna(row.get('pais_obra')) else str(row.get('pais_obra')),
            int(row['publico_total']) if pd.notnull(row['publico_total']) else None,
            float(row['renda_total']) if pd.notnull(row['renda_total']) else None,
            None if pd.isna(row.get('distribuidora')) else str(row.get('distribuidora')),
            None if pd.isna(row.get('registro_distribuidora')) else str(row.get('registro_distribuidora')),
            None if pd.isna(row.get('cnpj_distribuidora')) else str(row.get('cnpj_distribuidora')),
            int(row['ano_lancamento']) if pd.notnull(row['ano_lancamento']) else None,
            int(row['mes_lancamento']) if pd.notnull(row['mes_lancamento']) else None,
            int(row['dia_lancamento']) if pd.notnull(row['dia_lancamento']) else None,
            bool(row['is_outlier']) if 'is_outlier' in row.index and pd.notnull(row['is_outlier']) else False
        )
        cur.execute(insert_sql, vals)
        inserted += 1

    print(f"Dados carregados com sucesso! Total inserido: {inserted}")
    cur.close()
    conn.close()

except Exception as e:
    print(f"Erro ao conectar/popular o banco: {e}")

Estabelecendo conexão com o banco de dados...
Carregando DDL de: ../Data-Layer/silver/ddl.sql
Executando DDL para criar tabela 'lancamentos'...
✓ Tabela 'lancamentos' criada/verificada com sucesso!
✓ Índices criados com sucesso!
Carregando dados de sucesso no banco...
Dados carregados com sucesso! Total inserido: 6653
